# Verifisering og produksjon av OFV-inputfiler

Denne notebooken leser rå Excel-filer fra `RåData/<Selskap>/`, validerer og normaliserer innholdet, og skriver de tre eksakte inputfilene som [02_OFV_API_Kontroller](#notebook-1535288192161840) forventer i `Input/<Selskap>/`.

## Slik fungerer det
1. Notebooken scanner `RåData/` etter undermapper og viser dem som en **dropdown-widget**
2. Du fyller inn **org.nr.** og **kontrollperiode** i widget-feltene
3. Når du velger et selskap, scannes `RåData/<Selskap>/` etter `.xlsx`-filer
3. Filer **matches automatisk** til riktig kontroll basert på filnavn:
   * Filnavn med "solgt" eller "salg" → Kontroll solgte biler
   * Filnavn med "varekj" eller "brukt" → Varekjøp bruktbil
   * Filnavn med "demo" → Kontroll Demobil
4. Finnes flere filer som matcher samme kontroll, brukes den **nyeste** (sist endret)
5. Filer som ikke matcher noe mønster logges som "ukjent"

## Mappestruktur per selskap
* `RåData/<Selskap>/` — rå kildefiler
* `Input/<Selskap>/` — ferdig formaterte inputfiler
* `Output/<Selskap>/` — resultater fra OFV-kontroller

## Outputfiler som skrives til `Input/<Selskap>/`
* `kontroll_solgte_biler_input.xlsx`
* `kontroll_varekjop_bruktbil_input.xlsx`
* `kontroll_demobil_input.xlsx`

Hver output-fil får to faner:
* `Konfig` med kolonnene `Felt`, `Verdi`
* `Biler` med kontrollspesifikke kolonner i riktig format

In [ ]:
%pip install openpyxl -q

In [ ]:
# Midlertidig celle - vis kolonner i råfilene for mapping
import pandas as pd, os

raw_dir = "/Workspace/Users/stefan.luidold@bdo.no/OFV API/RåData/Test 1"
for fname in sorted(os.listdir(raw_dir)):
    if not fname.lower().endswith((".xlsx", ".xls")):
        continue
    path = os.path.join(raw_dir, fname)
    xls = pd.ExcelFile(path, engine="openpyxl")
    print(f"\n{'='*60}")
    print(f"{fname}")
    print(f"Arkfaner: {xls.sheet_names}")
    for sheet in xls.sheet_names[:3]:
        df = pd.read_excel(xls, sheet_name=sheet, nrows=2)
        print(f"\n  Fane '{sheet}' - {len(df.columns)} kolonner:")
        for col in df.columns:
            print(f"    - {col}")
        if len(df) > 0:
            print(f"  Eksempelrad: {dict(df.iloc[0])}")

In [ ]:
# ==================================================================
# CELLE 1: KONFIGURASJON - Selskapsvalg og filgjenkjenning
# ==================================================================

import os
import re
from datetime import date, datetime

import pandas as pd

_BASE = "/Workspace/Users/stefan.luidold@bdo.no/OFV API"
_RAW_ROOT = os.path.join(_BASE, "RåData")
os.makedirs(_RAW_ROOT, exist_ok=True)

# ------------------------------------------------------------------
# 1. Dropdown-widget: scan RåData/ etter undermapper (= selskaper)
# ------------------------------------------------------------------

_selskaper = sorted([
    d for d in os.listdir(_RAW_ROOT)
    if os.path.isdir(os.path.join(_RAW_ROOT, d)) and not d.startswith(".")
])

if not _selskaper:
    raise FileNotFoundError(
        f"Ingen undermapper funnet i {_RAW_ROOT}. "
        "Opprett en mappe med selskapsnavn og legg råfiler der."
    )

dbutils.widgets.dropdown("SELSKAP", _selskaper[0], _selskaper, "Velg selskap")
SELSKAP = dbutils.widgets.get("SELSKAP").strip()

print(f"Tilgjengelige selskaper: {_selskaper}")
print(f"Valgt selskap: {SELSKAP}")

# ------------------------------------------------------------------
# 1b. Konfig-parametere - gruppert per kontroll
#
#     Parameterne opprettes via notebook-parameterpanelet (ikke
#     dbutils.widgets.text) og leses med dbutils.widgets.get().
#     Navnekonvensjon: <Kontroll>_<Felt>
#
#     SOLGTE BILER:       Solgte_OrgNrSelger (valgfri)
#     VAREKJOP BRUKTBIL:  Varekjop_OrgNrKjoper, Varekjop_DatoFra,
#                         Varekjop_DatoTil (alle pakrevd)
#     DEMOBIL:            Demobil_OrgNr (pakrevd),
#                         Demobil_DatoFra, Demobil_DatoTil (valgfri)
# ------------------------------------------------------------------

# -- Les verdier fra notebook-parametere --
_W_ORG_SELGER   = dbutils.widgets.get("Solgte_OrgNrSelger").strip()
_W_ORG_KJOPER   = dbutils.widgets.get("Varekjop_OrgNrKjoper").strip()
_W_VAREKJOP_FRA = dbutils.widgets.get("Varekjop_DatoFra").strip()
_W_VAREKJOP_TIL = dbutils.widgets.get("Varekjop_DatoTil").strip()
_W_ORG_DEMOBIL  = dbutils.widgets.get("Demobil_OrgNr").strip()
_W_DEMOBIL_FRA  = dbutils.widgets.get("Demobil_DatoFra").strip()
_W_DEMOBIL_TIL  = dbutils.widgets.get("Demobil_DatoTil").strip()

print(f"\nKonfig-parametere:")
print(f"  [Solgte biler]")
print(f"    OrgNrSelger:  {_W_ORG_SELGER or '(tom - valgfri)'}")
print(f"  [Varekjop bruktbil]")
print(f"    OrgNrKjoper:  {_W_ORG_KJOPER or '(tom)'}")
print(f"    DatoFra:      {_W_VAREKJOP_FRA or '(tom)'}")
print(f"    DatoTil:      {_W_VAREKJOP_TIL or '(tom)'}")
print(f"  [Demobil]")
print(f"    OrgNr:        {_W_ORG_DEMOBIL or '(tom)'}")
print(f"    DatoFra:      {_W_DEMOBIL_FRA or '(tom - valgfri)'}")
print(f"    DatoTil:      {_W_DEMOBIL_TIL or '(tom - valgfri)'}")

# ------------------------------------------------------------------
# 2. Workspace-stier - per selskap
# ------------------------------------------------------------------

RAW_DIR = os.path.join(_RAW_ROOT, SELSKAP)
INPUT_DIR = os.path.join(_BASE, "Input", SELSKAP)
OUTPUT_DIR = os.path.join(_BASE, "Output", SELSKAP)

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"RåData:  {RAW_DIR}")
print(f"Input:   {INPUT_DIR}")
print(f"Output:  {OUTPUT_DIR}")

OUTPUT_FILE_SOLGTE = os.path.join(INPUT_DIR, "kontroll_solgte_biler_input.xlsx")
OUTPUT_FILE_VAREKJOP = os.path.join(INPUT_DIR, "kontroll_varekjop_bruktbil_input.xlsx")
OUTPUT_FILE_DEMOBIL = os.path.join(INPUT_DIR, "kontroll_demobil_input.xlsx")

# ------------------------------------------------------------------
# 3. Automatisk filgjenkjenning i RåData/<Selskap>/
#
#    Regler (case-insensitive mot filnavn uten endelse):
#      solgte_biler:      filnavn inneholder "solgt" eller "salg"
#      varekjop_bruktbil: filnavn inneholder "varekj" eller "brukt"
#      demobil:           filnavn inneholder "demo"
#
#    Treffer flere filer på samme kontroll, brukes den NYESTE (sist
#    endret). Filer som ikke matcher noe mønster listes som "ukjent".
# ------------------------------------------------------------------

_MATCH_RULES = {
    "solgte_biler":      [r"solgt", r"salg"],
    "varekjop_bruktbil": [r"varekj", r"brukt"],
    "demobil":           [r"demo"],
}


def _scan_raw_files(raw_dir: str) -> list:
    """Returnerer liste med (filnavn, full_sti, mtime) for alle
    .xlsx/.xls-filer i mappen (ikke undermapper)."""
    files = []
    for fname in os.listdir(raw_dir):
        fpath = os.path.join(raw_dir, fname)
        if os.path.isfile(fpath) and fname.lower().endswith((".xlsx", ".xls")):
            files.append((fname, fpath, os.path.getmtime(fpath)))
    return files


def _match_file_to_control(filename: str) -> str:
    """Returnerer kontrollnavn eller 'ukjent'."""
    stem = os.path.splitext(filename)[0].lower()
    for control_name, patterns in _MATCH_RULES.items():
        for pattern in patterns:
            if re.search(pattern, stem):
                return control_name
    return "ukjent"


def discover_files(raw_dir: str) -> dict:
    """Scanner mappen og returnerer {kontrollnavn: full_sti} for den
    nyeste filen som matcher hver kontroll. Ukjente filer logges."""
    all_files = _scan_raw_files(raw_dir)
    if not all_files:
        raise FileNotFoundError(
            f"Ingen Excel-filer (.xlsx/.xls) funnet i {raw_dir}"
        )

    # Grupper per kontrollnavn
    buckets = {}  # control_name -> [(fname, path, mtime), ...]
    ukjente = []
    for fname, fpath, mtime in all_files:
        ctrl = _match_file_to_control(fname)
        if ctrl == "ukjent":
            ukjente.append(fname)
        else:
            buckets.setdefault(ctrl, []).append((fname, fpath, mtime))

    # Velg nyeste fil per kontroll
    result = {}
    for ctrl, hits in buckets.items():
        hits.sort(key=lambda x: x[2], reverse=True)  # nyeste forst
        valgt_fname, valgt_path, _ = hits[0]
        result[ctrl] = valgt_path
        if len(hits) > 1:
            andre = [h[0] for h in hits[1:]]
            print(f"  {ctrl}: {valgt_fname}  (ignorerte eldre: {andre})")
        else:
            print(f"  {ctrl}: {valgt_fname}")

    if ukjente:
        print(f"  Ukjente filer (ikke matchet): {ukjente}")

    return result


print(f"\nScanner {RAW_DIR} ...")
DISCOVERED_FILES = discover_files(RAW_DIR)

_mangler = [c for c in _MATCH_RULES if c not in DISCOVERED_FILES]
if _mangler:
    print(f"\nADVARSEL: Ingen råfil funnet for: {_mangler}")
    print("  Disse kontrollene hoppes over.")

# ------------------------------------------------------------------
# 4. Validering av pakrevde konfig-parametere
#
#    Sjekker hvilke kontroller som har en matchet rafil og validerer
#    at de tilhorende widget-feltene er fylt ut for suksess.
#    - Org.nr.: ma vaere 9 siffer
#    - Datoer:  ma vaere gyldig dd.mm.aaaa
# ------------------------------------------------------------------

def _validate_orgnr(value: str, label: str) -> list:
    """Returnerer feilmelding(er) hvis org.nr. ikke er gyldig."""
    if not value:
        return [f"{label} er pakrevd men ikke fylt ut."]
    digits_only = value.replace(" ", "")
    if not digits_only.isdigit() or len(digits_only) != 9:
        return [f"{label} ma vaere 9 siffer (fikk: '{value}')."]
    return []


def _validate_date(value: str, label: str) -> list:
    """Returnerer feilmelding(er) hvis dato ikke er gyldig dd.mm.aaaa."""
    if not value:
        return [f"{label} er pakrevd men ikke fylt ut."]
    text = value.strip().replace("/", ".").replace("-", ".")
    parts = text.split(".")
    if len(parts) != 3:
        return [f"{label} er ikke en gyldig dato (fikk: '{value}'). Bruk dd.mm.aaaa."]
    try:
        d, m, y = int(parts[0]), int(parts[1]), int(parts[2])
        if y < 100:
            y += 2000
        date(y, m, d)  # kaster ValueError hvis ugyldig
    except (ValueError, TypeError):
        return [f"{label} er ikke en gyldig dato (fikk: '{value}'). Bruk dd.mm.aaaa."]
    return []


_validation_errors = []

# -- Varekjop bruktbil: OrgNrKjoper + DatoFra + DatoTil er pakrevd
if "varekjop_bruktbil" in DISCOVERED_FILES:
    _validation_errors += _validate_orgnr(_W_ORG_KJOPER, "OrgNrKjoper (varekjop bruktbil)")
    _validation_errors += _validate_date(_W_VAREKJOP_FRA, "DatoFra (varekjop bruktbil)")
    _validation_errors += _validate_date(_W_VAREKJOP_TIL, "DatoTil (varekjop bruktbil)")

# -- Demobil: OrgNr er pakrevd, datoer valgfrie men valideres hvis utfylt
if "demobil" in DISCOVERED_FILES:
    _validation_errors += _validate_orgnr(_W_ORG_DEMOBIL, "OrgNr (demobil)")
    if _W_DEMOBIL_FRA:
        _validation_errors += _validate_date(_W_DEMOBIL_FRA, "DatoFra (demobil)")
    if _W_DEMOBIL_TIL:
        _validation_errors += _validate_date(_W_DEMOBIL_TIL, "DatoTil (demobil)")

# -- Solgte biler: OrgNrSelger er valgfri, men valideres hvis fylt ut
if "solgte_biler" in DISCOVERED_FILES and _W_ORG_SELGER:
    _validation_errors += _validate_orgnr(_W_ORG_SELGER, "OrgNrSelger (solgte biler)")

# -- Dato-kryssvalidering per kontroll: DatoFra ma vaere for DatoTil
def _check_date_order(fra_val, til_val, label):
    if not fra_val or not til_val:
        return []
    try:
        _p = fra_val.replace("/",".").replace("-",".").split(".")
        d1 = date(int(_p[2]) if int(_p[2]) > 99 else int(_p[2])+2000, int(_p[1]), int(_p[0]))
        _p = til_val.replace("/",".").replace("-",".").split(".")
        d2 = date(int(_p[2]) if int(_p[2]) > 99 else int(_p[2])+2000, int(_p[1]), int(_p[0]))
    except Exception:
        return []
    if d1 > d2:
        return [f"{label}: DatoFra ({fra_val}) er etter DatoTil ({til_val})."]
    return []

_validation_errors += _check_date_order(_W_VAREKJOP_FRA, _W_VAREKJOP_TIL, "Varekjop bruktbil")
_validation_errors += _check_date_order(_W_DEMOBIL_FRA, _W_DEMOBIL_TIL, "Demobil")

if _validation_errors:
    print("\n" + "=" * 60)
    print("VALIDERINGSFEIL - fyll ut widget-feltene og kjor cellen pa nytt:")
    print("=" * 60)
    for err in _validation_errors:
        print(f"  FEIL: {err}")
    raise ValueError(
        f"{len(_validation_errors)} valideringsfeil i konfig-parametere. "
        "Se utskrift over for detaljer."
    )
else:
    print("\nKonfig-validering OK")

# ------------------------------------------------------------------
# 5. Kontroll-mapping (kolonner, konfig) - UAVHENGIG av filnavn.
#    Tilpass rename_map, konfig, osv. til råfilenes kolonnestruktur.
#    source_sheet angir arkfane (default 0 = forste ark).
# ------------------------------------------------------------------

# ------------------------------------------------------------------
# TILPASS DENNE SEKSJONEN TIL DINE RÅFILER.
# Bruk diagnostikk-cellen over for å se hvilke kolonner råfilene har.
#
# Råfiler i "Test 1" som ble oppdaget:
#   solgte_biler:      Regnr, Vin, Bokfort
#   varekjop_bruktbil: Avdeling, Selger, KontraktsID, Masternr., Reg.nr.
#                      (OBS: mangler Liste-kolonne - alle rader settes til Innkjop)
#   demobil:           Beskrivelse, Regnr / VIN-nr, Avsk.hittil, ...
# ------------------------------------------------------------------

CONTROL_MAPPINGS = {
    "solgte_biler": {
        "source_sheet": 0,
        "identifier_col": None,
        "regno_col": "Regnr",
        "vin_col": "Vin",
        "rename_map": {
            "Bokfort": "BokfortDato",
        },
        "required_biler_columns": ["BokfortDato"],
        "konfig": {
            "OrgNrSelger": _W_ORG_SELGER,
        },
    },
    "varekjop_bruktbil": {
        "source_sheet": 0,
        "identifier_col": None,
        "regno_col": "Reg.nr.",
        "vin_col": "_ingen_vin_kolonne_",
        "rename_map": {},
        "required_biler_columns": ["Liste"],
        "konfig": {
            "OrgNrKjoper": _W_ORG_KJOPER,
            "DatoFra": _W_VAREKJOP_FRA,
            "DatoTil": _W_VAREKJOP_TIL,
        },
        "allowed_liste_values": {"innkjop": "Innkjop", "ib": "IB", "ub": "UB"},
        # Hvis råfilen mangler "Liste"-kolonne, legges den til automatisk
        # med defaultverdi "Innkjop" i prepare_biler_df.
        "default_liste": "Innkjop",
    },
    "demobil": {
        "source_sheet": 0,
        "identifier_col": "Regnr / VIN-nr",
        "regno_col": None,
        "vin_col": None,
        "rename_map": {
            "Avsk.hittil": "BokfortInnDato",
        },
        "required_biler_columns": ["BokfortInnDato"],
        "konfig": {
            "OrgNr": _W_ORG_DEMOBIL,
            "DatoFra": _W_DEMOBIL_FRA,
            "DatoTil": _W_DEMOBIL_TIL,
        },
    },
}

# ------------------------------------------------------------------
# 6. Rapport-innstillinger
# ------------------------------------------------------------------

WRITE_VALIDATION_REPORT = True
VALIDATION_REPORT_HTML = os.path.join(OUTPUT_DIR, "verifisering_input_rapport.html")

In [ ]:
# ==================================================================
# CELLE 2: HJELPEFUNKSJONER
# ==================================================================


def normalize_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return str(value).strip()


def normalize_identifier(value) -> str:
    return normalize_text(value).upper().replace(" ", "")


def is_vin(value: str) -> bool:
    return len(normalize_identifier(value)) == 17


def parse_user_date(raw):
    if raw is None:
        return None
    if isinstance(raw, float) and pd.isna(raw):
        return None
    if isinstance(raw, pd.Timestamp):
        return raw.date()
    if isinstance(raw, datetime):
        return raw.date()
    if isinstance(raw, date):
        return raw

    text = normalize_text(raw)
    if not text:
        return None
    text = text.replace("/", ".").replace("-", ".")
    parts = text.split(".")
    if len(parts) != 3:
        return None
    try:
        day, month, year = int(parts[0]), int(parts[1]), int(parts[2])
    except ValueError:
        return None
    if year < 100:
        year += 2000
    try:
        return date(year, month, day)
    except ValueError:
        return None


def build_identifikator(df: pd.DataFrame, spec: dict) -> pd.Series:
    identifier_col = spec.get("identifier_col")
    regno_col = spec.get("regno_col")
    vin_col = spec.get("vin_col")

    if identifier_col and identifier_col in df.columns:
        return df[identifier_col].map(normalize_identifier)

    reg_series = df[regno_col].map(normalize_identifier) if regno_col in df.columns else pd.Series([""] * len(df))
    vin_series = df[vin_col].map(normalize_identifier) if vin_col in df.columns else pd.Series([""] * len(df))

    values = []
    for regno, vin in zip(reg_series, vin_series):
        values.append(vin if is_vin(vin) else regno)
    return pd.Series(values, index=df.index)


def ensure_columns_exist(df: pd.DataFrame, columns: list, context: str) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f"Mangler kolonner i {context}: {missing}")


def read_raw_sheet(file_path: str, spec: dict) -> pd.DataFrame:
    """Leser råfil fra gitt sti. source_sheet fra spec (index eller navn)."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Fant ikke råfil: {file_path}")
    return pd.read_excel(file_path, sheet_name=spec.get("source_sheet", 0))


def resolve_konfig_value(df: pd.DataFrame, raw_value):
    if isinstance(raw_value, str) and raw_value.startswith("col:"):
        column_name = raw_value[4:]
        if column_name not in df.columns:
            raise ValueError(f"Konfig peker på kolonne som ikke finnes: {column_name}")
        non_null = df[column_name].dropna()
        return non_null.iloc[0] if len(non_null) else ""
    return raw_value


def build_konfig_df(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    rows = []
    for felt, raw_value in spec.get("konfig", {}).items():
        value = resolve_konfig_value(df, raw_value)
        rows.append({"Felt": felt, "Verdi": value})
    return pd.DataFrame(rows)


def normalize_liste_value(value: str, allowed_map: dict) -> str:
    raw = normalize_text(value).lower()
    if raw not in allowed_map:
        raise ValueError(f"Ugyldig Liste-verdi: '{value}'. Tillatt: {sorted(allowed_map.values())}")
    return allowed_map[raw]


def prepare_biler_df(control_name: str, df_raw: pd.DataFrame, spec: dict) -> pd.DataFrame:
    df = df_raw.copy()
    rename_map = spec.get("rename_map", {})
    df = df.rename(columns=rename_map)
    df["Identifikator"] = build_identifikator(df, spec)
    df = df[df["Identifikator"].astype(str).str.len() > 0].copy()

    # For varekjop: legg til default Liste FOR ensure_columns_exist
    if control_name == "varekjop_bruktbil" and "Liste" not in df.columns:
        default_liste = spec.get("default_liste", "Innkjop")
        df["Liste"] = default_liste
        print(f"  OBS: 'Liste'-kolonne mangler i rafil - alle rader satt til '{default_liste}'")

    required = ["Identifikator"] + spec.get("required_biler_columns", [])
    ensure_columns_exist(df, required, control_name)

    if control_name == "solgte_biler":
        df["BokfortDato"] = df["BokfortDato"].map(parse_user_date)
        biler = df[["Identifikator", "BokfortDato"]].copy()

    elif control_name == "varekjop_bruktbil":
        allowed_map = spec.get("allowed_liste_values", {"innkjop": "Innkjop", "ib": "IB", "ub": "UB"})
        # Hvis Liste allerede ble satt med default (over), normaliser bare eksisterende verdier
        if df["Liste"].nunique() == 1 and df["Liste"].iloc[0] in allowed_map.values():
            pass  # allerede satt som default
        else:
            df["Liste"] = df["Liste"].map(lambda x: normalize_liste_value(x, allowed_map))
        biler = df[["Identifikator", "Liste"]].copy()

    elif control_name == "demobil":
        df["BokfortInnDato"] = df["BokfortInnDato"].map(parse_user_date)
        biler = df[["Identifikator", "BokfortInnDato"]].copy()

    else:
        raise ValueError(f"Ukjent kontroll: {control_name}")

    biler = biler.drop_duplicates().reset_index(drop=True)
    return biler


def validate_biler_df(control_name: str, biler_df: pd.DataFrame) -> list:
    messages = []
    if biler_df.empty:
        messages.append(f"{control_name}: Ingen gyldige biler etter transformasjon")

    if "Identifikator" in biler_df.columns:
        blank_count = (biler_df["Identifikator"].astype(str).str.strip() == "").sum()
        if blank_count:
            messages.append(f"{control_name}: {blank_count} tomme identifikatorer")

    if control_name == "solgte_biler":
        null_dates = biler_df["BokfortDato"].isna().sum()
        if null_dates:
            messages.append(f"{control_name}: {null_dates} rader mangler BokfortDato")

    if control_name == "demobil":
        null_dates = biler_df["BokfortInnDato"].isna().sum()
        if null_dates:
            messages.append(f"{control_name}: {null_dates} rader mangler BokfortInnDato")

    return messages


def write_control_workbook(output_path: str, konfig_df: pd.DataFrame, biler_df: pd.DataFrame) -> None:
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        konfig_df.to_excel(writer, sheet_name="Konfig", index=False)
        biler_df.to_excel(writer, sheet_name="Biler", index=False)


def write_validation_report(rows: list) -> None:
    html_rows = "".join(
        f"<tr><td>{r['kontroll']}</td><td>{r['source_file']}</td><td>{r['rows_in']}</td><td>{r['rows_out']}</td><td>{r['status']}</td><td>{r['message']}</td></tr>"
        for r in rows
    )
    html = f"""
    <html>
      <head><meta charset='utf-8'><title>Verifisering inputrapport</title></head>
      <body>
        <h1>Verifisering av OFV-inputfiler</h1>
        <table border='1' cellspacing='0' cellpadding='6'>
          <tr><th>Kontroll</th><th>Kildefil</th><th>Rader inn</th><th>Rader ut</th><th>Status</th><th>Melding</th></tr>
          {html_rows}
        </table>
      </body>
    </html>
    """
    with open(VALIDATION_REPORT_HTML, "w", encoding="utf-8") as f:
        f.write(html)

In [ ]:
# ==================================================================
# CELLE 3: BYGG DE TRE INPUTFILENE
# ==================================================================

BUILD_PLAN = [
    {"control_name": "solgte_biler",      "output_path": OUTPUT_FILE_SOLGTE},
    {"control_name": "varekjop_bruktbil", "output_path": OUTPUT_FILE_VAREKJOP},
    {"control_name": "demobil",           "output_path": OUTPUT_FILE_DEMOBIL},
]

build_results = []
validation_messages = []

for item in BUILD_PLAN:
    control_name = item["control_name"]
    output_path = item["output_path"]

    # Hent auto-oppdaget filsti fra Celle 1
    source_path = DISCOVERED_FILES.get(control_name)

    if source_path is None:
        build_results.append({
            "kontroll": control_name,
            "source_file": "(ingen fil funnet)",
            "output_file": output_path,
            "rows_in": 0,
            "rows_out": 0,
            "status": "Hoppet over",
            "message": f"Ingen råfil matchet kontroll '{control_name}' i {RAW_DIR}",
        })
        continue

    spec = CONTROL_MAPPINGS[control_name]

    try:
        raw_df = read_raw_sheet(source_path, spec)
        konfig_df = build_konfig_df(raw_df, spec)
        biler_df = prepare_biler_df(control_name, raw_df, spec)
        messages = validate_biler_df(control_name, biler_df)
        validation_messages.extend(messages)
        write_control_workbook(output_path, konfig_df, biler_df)

        build_results.append({
            "kontroll": control_name,
            "source_file": source_path,
            "output_file": output_path,
            "rows_in": len(raw_df),
            "rows_out": len(biler_df),
            "status": "OK" if not messages else "Varsel",
            "message": "; ".join(messages) if messages else "Filen ble bygd uten valideringsfeil",
        })

    except Exception as exc:
        build_results.append({
            "kontroll": control_name,
            "source_file": source_path,
            "output_file": output_path,
            "rows_in": 0,
            "rows_out": 0,
            "status": "Feil",
            "message": str(exc),
        })

build_results_df = pd.DataFrame(build_results)
build_results_df

In [ ]:
# ==================================================================
# CELLE 4: RAPPORT OG OPPSUMMERING
# ==================================================================

if WRITE_VALIDATION_REPORT:
    write_validation_report(build_results)
    print(f"Valideringsrapport lagret: {VALIDATION_REPORT_HTML}")

print("=" * 70)
print("VERIFISERING AV OFV-INPUT")
print("=" * 70)

for row in build_results:
    print(f"--- {row['kontroll']} ---")
    print(f"  Kilde: {row['source_file']}")
    print(f"  Output: {row['output_file']}")
    print(f"  Rader inn: {row['rows_in']}")
    print(f"  Rader ut: {row['rows_out']}")
    print(f"  Status: {row['status']}")
    print(f"  Melding: {row['message']}")
    print()

if validation_messages:
    print("Valideringsmeldinger:")
    for msg in validation_messages:
        print(f"  - {msg}")
else:
    print("Ingen valideringsmeldinger.")

## Slik bruker du notebooken

1. Opprett en undermappe i [RåData](#folder-1535288192161839) med selskapets navn (f.eks. `Toyota Bilia Oslo AS`)
2. Legg rå Excel-filer i undermappen. Navngi dem slik at filnavnet inneholder:
   * `solgt` eller `salg` for solgte biler
   * `varekj` eller `brukt` for varekjøp bruktbil
   * `demo` for demobiler
3. Kjør Celle 1 — velg riktig selskap i dropdown-widgeten, se at filene matches korrekt
4. Kjør Celle 2-4
5. Kontroller at de tre inputfilene er opprettet i `Input/<Selskap>/`
6. Kjør [02_OFV_API_Kontroller](#notebook-1535288192161840) med samme selskapsnavn

## Tips
* Filnavn trenger ikke være eksakte — bare ha riktig nøkkelord et sted i navnet
* Hvis du har flere filer som matcher samme kontroll, brukes den sist endrede
* `CONTROL_MAPPINGS` i Celle 1 må fortsatt tilpasses hvis råfilene har andre kolonnenavn enn standardoppsettet